# Benchmarking VectorRAG + GraphRAG Recommendation System
This notebook runs recall@5 benchmarks for tool and workflow retrieval using the BGE model and the graph-xp pipelines.

In [21]:
# Setup imports and project root
from pathlib import Path
import sys
import json
import yaml
from neo4j import GraphDatabase

# Robust project root detection (look for agents directory)
cwd = Path.cwd()
PROJECT_ROOT = None
for parent in [cwd] + list(cwd.parents):
    if (parent / "agents").exists():
        PROJECT_ROOT = parent
        break
if PROJECT_ROOT is None:
    raise RuntimeError("Project root not found")

sys.path.insert(0, str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)

# Paths
config_path = PROJECT_ROOT / "agents/graph-xp/config/graph_db_config.yml"
tool_queries_path = PROJECT_ROOT / "agents/graph-xp/benchmarks/tool_test_queries.json"
workflow_queries_path = PROJECT_ROOT / "agents/graph-xp/benchmarks/workflow_test_queries.json"

print("Config exists:", config_path.exists())
print("Tool queries exists:", tool_queries_path.exists())
print("Workflow queries exists:", workflow_queries_path.exists())

with open(config_path, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f) or {}
neo_cfg = cfg.get("neo4j") or {}
print("Neo4j URI:", neo_cfg.get("uri"))


Project root: /home/henok/Desktop/galaxy-agent-xp-II
Config exists: True
Tool queries exists: True
Workflow queries exists: True
Neo4j URI: bolt://localhost:7687


In [22]:
# Initialize clients and pipelines (load modules by file path to avoid package name issues)
import importlib.util
from pathlib import Path
from neo4j import GraphDatabase
import sys

# ensure graph-xp dir on sys.path so its internal relative imports work
graph_xp_dir = PROJECT_ROOT / "agents/graph-xp"
if str(graph_xp_dir) not in sys.path:
    sys.path.insert(0, str(graph_xp_dir))

def load_module(mod_name: str, file_path: Path):
    spec = importlib.util.spec_from_file_location(mod_name, file_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

tool_pipeline_mod = load_module(
    "tool_retrieval_pipeline",
    graph_xp_dir / "pipeline/tool_retrieval_pipeline.py",
 )
workflow_pipeline_mod = load_module(
    "workflow_retrival_pipeline",
    graph_xp_dir / "pipeline/workflow_retrival_pipeline.py",
 )
embedder_mod = load_module(
    "query_embedding",
    graph_xp_dir / "scripts/query_embedding.py",
 )

ToolRetrievalPipeline = tool_pipeline_mod.ToolRetrievalPipeline
WorkflowRetrievalPipeline = workflow_pipeline_mod.WorkflowRetrievalPipeline
QueryEmbeddingService = embedder_mod.QueryEmbeddingService

class SimpleNeoClient:
    """Minimal Neo4j client with run_query + driver for pipelines."""
    def __init__(self, uri: str, user: str, password: str):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))
    def run_query(self, cypher: str, parameters=None):
        with self.driver.session() as session:
            result = session.run(cypher, **(parameters or {}))
            return [dict(r) for r in result]

neo_client = SimpleNeoClient(
    uri=neo_cfg.get("uri"),
    user=neo_cfg.get("user"),
    password=neo_cfg.get("password"),
 )
tool_pipeline = ToolRetrievalPipeline(neo_client)
workflow_pipeline = WorkflowRetrievalPipeline(neo_client)
embedder = QueryEmbeddingService()

print("Connected to Neo4j at", neo_cfg.get("uri"))
print("Loaded embedding model:", embedder.model_name)

 Loaded embedding model: BAAI/bge-base-en-v1.5
 Loaded embedding model: BAAI/bge-base-en-v1.5
 Loaded embedding model: BAAI/bge-base-en-v1.5
Connected to Neo4j at bolt://localhost:7687
Loaded embedding model: BAAI/bge-base-en-v1.5


In [24]:
# Tool retrieval benchmark (recall@k)
from sklearn.metrics.pairwise import cosine_similarity


def compute_tool_recall_with_embedding(tool_pipeline, queries, ground_truth, embedder, k=5, threshold=0.8):
    recall_scores = []
    total_recall = 0

    for query in queries:
        results = tool_pipeline.retrieve_tools(query, top_k=k)
        retrieved_items = []
        for ctx in results[:k]:
            tool = ctx.get("tool", {})
            name = tool.get("display_name") or tool.get("tool_name") or tool.get("name") or tool.get("tool_id")
            if name:
                retrieved_items.append(name)
        retrieved_items = list(set(retrieved_items))
        expected_items = ground_truth.get(query, [])

        if not expected_items or not retrieved_items:
            recall = 0
        else:
            expected_embeds = embedder.embed_query(expected_items)
            retrieved_embeds = embedder.embed_query(retrieved_items)
            sim_matrix = cosine_similarity(expected_embeds, retrieved_embeds)
            max_sims = sim_matrix.max(axis=1)
            recall = 1 if any(s >= threshold for s in max_sims) else 0

        recall_scores.append(recall)
        total_recall += recall

        print(f"\nQuery: {query}")
        print(f"Expected: {expected_items}")
        print(f"Top-{k} retrieved: {retrieved_items}")
        print(f"Recall@{k} : {recall}")

    avg_recall = total_recall / len(queries)
    print(f"\nAverage Recall@{k} : {avg_recall:.2f}")
    return recall_scores, avg_recall



In [25]:
with open(tool_queries_path, "r") as f:
    tool_dataset = json.load(f)

tool_queries = tool_dataset["queries"]
tool_ground_truth = tool_dataset["ground_truth"]

print(f"Loaded {len(tool_queries)} tool queries")
tool_recall_scores, tool_avg_recall = compute_tool_recall_with_embedding(
    tool_pipeline=tool_pipeline,
    queries=tool_queries,
    ground_truth=tool_ground_truth,
    embedder=embedder,
    k=5,
    threshold=0.8,
)

Loaded 25 tool queries

Query: I want to search a sequence database for a query sequence using jackhmmer
Expected: ['jackhmmer']
Top-5 retrieved: ['Filter sequences by ID', 'Search ChEMBL database', 'Select Sequences']
Recall@5 : 0

Query: tool to retrieve genomic datasets for completed Microbial Genome Projects from NCBI
Expected: ['Get Microbial Data']
Top-5 retrieved: ['NCBI Datasets Genomes', 'Get Microbial Data']
Recall@5 : 1

Query: convert genome coordinates or annotation files between different assembly versions
Expected: ['CrossMap Wig']
Top-5 retrieved: ['Converts genome bins in fasta format', 'Convert genome coordinates']
Recall@5 : 0

Query: how can I translate gene identifiers between different organisms
Expected: ['gProfiler Orth']
Top-5 retrieved: ['Gene BED To Exon/Intron/Codon BED', 'Translate nucleotides', 'Replace ambiguous codons', 'Renumber GenBank Genes']
Recall@5 : 0

Query: tool to download a list of URLs via lftp and create a collection
Expected: ['downloads']


In [ ]:
# Workflow retrieval benchmark (recall@k)
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np


def pick_workflow_name(wf: dict) -> str | None:
    """Choose the most human-readable name available for a workflow."""
    for key in ("display_name", "name", "workflow_name", "file_name", "workflow_repository", "workflow_id"):
        val = wf.get(key)
        if val:
            return val
    return None


def normalize_name(name: str | None) -> str | None:
    """Strip common file extensions so names match ground truth strings."""
    if not name:
        return name
    lowered = name.lower()
    for ext in (".ga", ".gxwf", ".json"):
        if lowered.endswith(ext):
            return name[: -len(ext)]
    return name


def compute_workflow_recall_with_embedding(workflow_pipeline, queries, ground_truth, embedder, k=5, threshold=0.8):
    recall_scores = []
    total_recall = 0

    for query in queries:
        results = workflow_pipeline.retrieve_workflows(query, top_k=k)
        retrieved_names = []
        for ctx in results[:k]:
            wf = ctx.get("workflow", {})
            name = normalize_name(pick_workflow_name(wf))
            if name:
                retrieved_names.append(name)
        retrieved_names = list(set(retrieved_names))
        expected_names = [normalize_name(n) for n in ground_truth.get(query, [])]

        if not expected_names or not retrieved_names:
            recall = 0
        else:
            expected_embeds = embedder.embed_query(expected_names)
            retrieved_embeds = embedder.embed_query(retrieved_names)
            sim_matrix = cosine_similarity(expected_embeds, retrieved_embeds)
            max_sims = sim_matrix.max(axis=1)
            recall = 1 if any(s >= threshold for s in max_sims) else 0

        recall_scores.append(recall)
        total_recall += recall

        print(f"\nQuery: {query}")
        print(f"Expected: {expected_names}")
        print(f"Top-{k} retrieved: {retrieved_names}")
        print(f"Recall@{k}: {recall}")

    avg_recall = total_recall / len(queries)
    print(f"\nAverage Recall@{k}: {avg_recall:.2f}")
    return recall_scores, avg_recall
    


In [28]:
with open(workflow_queries_path, "r") as f:
    wf_dataset = json.load(f)

wf_queries = wf_dataset["queries"]
wf_ground_truth = wf_dataset["ground_truth"]

print(f"Loaded {len(wf_queries)} workflow queries")
wf_recall_scores, wf_avg_recall = compute_workflow_recall_with_embedding(
    workflow_pipeline=workflow_pipeline,
    queries=wf_queries,
    ground_truth=wf_ground_truth,
    embedder=embedder,
    k=5,
    threshold=0.8,
)

Loaded 25 workflow queries

Query: I want a genome assembly workflow using HiFi reads with HiC phasing following VGP4 standards
Expected: ['Genome Assembly from Hifi reads with HiC phasing - VGP4']
Top-5 retrieved: ['Assembly-Hifi-HiC-phasing-VGP4', 'Assembly-Hifi-only-VGP3', 'Assembly-Hifi-Trio-phasing-VGP5', 'hi-c-map-for-assembly-manual-curation', 'Scaffolding-HiC-VGP8']
Recall@5: 1

Query: workflow for scaffolding a genome using Bionano optical map data
Expected: ['Scaffolding-BioNano-VGP7']
Top-5 retrieved: ['Scaffolding-BioNano-VGP7', 'Assembly-Hifi-HiC-phasing-VGP4', 'Mitogenome-Assembly-VGP0', 'hi-c-map-for-assembly-manual-curation', 'Scaffolding-HiC-VGP8']
Recall@5: 1

Query: how to perform k-mer profiling for PacBio HiFi trio data for VGP2
Expected: ['kmer-profiling-hifi-trio-VGP2']
Top-5 retrieved: ['kmer-profiling-hifi-trio-VGP2', 'Assembly-Hifi-HiC-phasing-VGP4', 'Assembly-Hifi-Trio-phasing-VGP5', 'Assembly-Hifi-only-VGP3', 'kmer-profiling-hifi-VGP1']
Recall@5: 1

Query: d